# `index(base, start, end=None, quote=None)`

Daily levels for the market indexes `equity_indexes.prices` holds, from FMP, as stored or in any currency `forex()` can price. Every cell runs against prod, read-only, as `readonly_user`. Change the arguments and re-run any cell.

In [ ]:
from datetime import date, timedelta

import polars as pl
from IPython.display import Markdown

from arora_data.dates import today_et
from arora_data.frames import to_df
from arora_data.index import INDEXES, IndexBar
from arora_data.sync import index, list_series


def show(rows):
    """One result or a list of results as a Markdown table."""
    rows = [r.model_dump() for r in (rows if isinstance(rows, list) else [rows])]
    if not rows:
        return Markdown("0 rows")
    head = list(rows[0])
    lines = ["|".join(head), "|".join("---" for _ in head)]
    lines += ["|".join(str(r[c]).replace("|", "/") for c in head) for r in rows]
    return Markdown(f"{len(rows)} rows\n\n" + "\n".join(lines))

## The indexes
Read from the table on every call, so an index the main repo adds or drops shows up here with no code change. The 55 the main repo ingests have a name and the currency their members trade in; the 10 volatility series have no currency. Any other stored index works by symbol, with `name` and `currency` blank.

In [ ]:
pl.DataFrame([{"index": s.key, "name": INDEXES.get(s.key, (None, None))[0], "currency": INDEXES.get(s.key, (None, None))[1],
               "latest_date": s.latest_date} for s in list_series("equity_indexes", "prices")])

## By symbol or name
The caret is optional, and a name or alias works anywhere a symbol does, in any case and spacing. Exchange-suffixed symbols such as `000001.SS` have no caret. Aliases include `dow`, `djia`, `tsx`, `taiex`, `asx 200`, `s&p 400` and `shanghai composite`.

In [ ]:
week = today_et() - timedelta(days=7)
assert index("^GSPC", week) == index("gspc", week) == index("S&P 500", week)
assert index("dow", week) == index("^DJI", week)
show(index("nikkei 225", week)[-2:] + index("shanghai composite", week)[-2:] + index("vix", week)[-2:])

## Levels as stored, the default
FMP daily bars as stored (`route` = stored): open, high, low, close, volume and VWAP, in the index's own currency (`currency`). `quote` and `quote_date` are blank because no rate is used.

In [ ]:
show(index("^GSPC", today_et() - timedelta(days=7)))

## In another currency
The stored level times `forex(currency, "USD")` that day (`route` = crossed): here the Nikkei 225 in US dollars. Open uses the day's opening rate, close its closing rate. High, low and VWAP are blank: the index and the rate peak at different moments. Volume stays as stored. A day the pair has no rate carries the last close, and `quote_date` names the rate's day.

In [ ]:
show(index("nikkei 225", today_et() - timedelta(days=7), quote="USD"))

An index already in the quote currency comes back as stored, so `quote="USD"` puts every index on one footing: the S&P 500 keeps its high, low and VWAP, the others are crossed.

In [ ]:
start = today_et() - timedelta(days=7)
rows = [r for q in ["USD", "EUR", "JPY", "GBP", "INR", "CHF"] for r in index("^GSPC", start, quote=q)]
to_df(rows, IndexBar).pivot(on="quote", index="date", values="close")

In [ ]:
world = ["^GSPC", "^FTSE", "^GDAXI", "^N225", "^HSI", "^NSEI", "^BVSP", "000001.SS"]
usd = to_df(index("ALL", today_et() - timedelta(days=7), quote="USD"), IndexBar)
usd.filter(pl.col("index").is_in(world)).pivot(on="index", index="date", values="close").sort("date")

## `index("ALL", start, end, quote=None)`
Every stored index, ordered by symbol then date. Each symbol is read once, and each currency's rate once. With a quote, the volatility series (no currency) are left out.

In [ ]:
to_df(index("ALL", today_et() - timedelta(days=7)), IndexBar).pivot(on="index", index="date", values="close")

## Full history in one call
The S&P 500 from the start of FMP's history. There is no row limit.

In [ ]:
spx = to_df(index("^GSPC", date(1900, 1, 1)), IndexBar)
print(f"{spx.height:,} days, {spx['date'].min()} to {spx['date'].max()}")
(
    spx.group_by(pl.col("date").dt.year().alias("year"), maintain_order=True)
    .agg(pl.col("date").last().alias("last_day"), pl.col("close").last())
    .tail(10)
)

## Volume is served as stored, and often meaningless
FMP reports zero volume every day for the volatility series, the Russell indexes but `^RUT`, `^W5000`, `^SPXEW` and `^SP500TR`; `^MID` is partly zeros, `^OEX` repeats the S&P 500's volume, and `^IBEX` repeats some days' volume. Here: the share of zero-volume days over the last year, for every index that has any.

In [ ]:
year = to_df(index("ALL", today_et() - timedelta(days=365)), IndexBar)
(
    year.group_by("index").agg((pl.col("volume") == 0).mean().round(3).alias("zero_volume_share"), pl.len().alias("days"))
    .filter(pl.col("zero_volume_share") > 0)
    .sort("zero_volume_share", descending=True)
)

In [ ]:
oex = year.filter(pl.col("index") == "^OEX").select("date", pl.col("volume").alias("oex"))
gspc = year.filter(pl.col("index") == "^GSPC").select("date", pl.col("volume").alias("gspc"))
both = oex.join(gspc, on="date")
print(f"^OEX volume equals ^GSPC volume on {(both['oex'] == both['gspc']).sum()} of {both.height} days")

## Real sessions and FMP placeholders
India's NSE holds special weekend sessions (Union Budget days, Diwali's Muhurat trading), so `^NSEI` has real weekend bars, with volume. Every other weekend bar is an FMP placeholder, flat (open = high = low = close) or with zero volume: the Russell indexes on Saturday 2025-04-26, `^TWII` on Sunday 2026-09-20, and Sundays on `^GSPC` and `^N225` in 1932–1956. Placeholders fall on weekdays too: across the full history FMP has 226 flat zero-volume weekday bars, most on `^GSPTSE` and `000001.SS`. The ones checked against FMP on 2026-11-08 are exactly what FMP serves, so all are served as stored. Here: every weekend bar over the last five years.

In [ ]:
five = to_df(index("ALL", today_et() - timedelta(days=5 * 365)), IndexBar)
five.filter(pl.col("date").dt.weekday() >= 6).select(
    "index", "date", pl.col("date").dt.strftime("%a").alias("weekday"), "open", "high", "low", "close", "volume")

## Long gaps are market closures
A gap of over a week between bars is a holiday closure (Golden Week, Lunar New Year, Chuseok), not missing data. Here: every gap of more than 7 days in the last five years.

In [ ]:
(
    five.sort("index", "date")
    .with_columns(pl.col("date").shift().over("index").alias("previous"))
    .with_columns((pl.col("date") - pl.col("previous")).dt.total_days().alias("gap_days"))
    .filter(pl.col("gap_days") > 7)
    .select("index", "previous", "date", "gap_days")
)

## Duplicates FMP lists
`^SPX` is the same series as `^GSPC`, so it returns `^GSPC`. `^DJX`, `^XSP`, `^NANOS` and `^MRUT` are scaled copies of `^DJI`, `^GSPC` and `^RUT`; they are not stored and raise an error naming the index to use.

In [ ]:
assert index("^SPX", week) == index("^GSPC", week)
for base in ["^DJX", "xsp", "^NANOS", "mrut"]:
    try:
        index(base, week)
    except ValueError as err:
        print(err)

## Bad input
A symbol FMP lists but the main repo doesn't ingest, a partial name, a quote for a volatility series (it has no currency), `quote="ALL"`, a currency `forex()` can't price, or a start after the end each raise a clear error.

In [ ]:
for base, quote in [("^TNX", None), ("^SOX", None), ("s&p", None), ("nasdaq", None), ("vix", "EUR"), ("^GSPC", "ALL"),
                    ("^GSPC", "XYZ")]:
    try:
        index(base, week, quote=quote)
    except ValueError as err:
        print(f"{base}/{quote}: {err}")
try:
    index("^GSPC", today_et(), week)
except ValueError as err:
    print(f"start after end: {err}")